In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, Dataset
import numpy as np
import pandas as pd
import random
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split


# Mappa classi
class_map = {
    'normal': 0, 'backdoor': 1, 'ddos': 2, 'dos': 3, 'injection': 4,
    'mitm': 5, 'password': 6, 'ransomware': 7, 'scanning': 8, 'xss': 9
}

class_map_inverse = {v: k for k, v in class_map.items()}

In [2]:
# ──────────────────────────────────────────────── Modello MLP
class SimpleMLP(nn.Module):
    def __init__(self, input_size, num_classes=10):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.fc(x)

# ──────────────────────────────────────────────── Replay Buffer
class ReplayBuffer:
    def __init__(self, max_size=4000):
        self.max_size = max_size
        self.buffer = []
        self.current_size = 0

    def add(self, features, labels, task_id):
        for feat, lbl in zip(features, labels):
            if self.current_size < self.max_size:
                self.buffer.append((feat, lbl, task_id))
                self.current_size += 1
            else:
                idx = random.randint(0, self.current_size - 1)
                self.buffer[idx] = (feat, lbl, task_id)

    def get_batch(self, batch_size):
        if len(self.buffer) == 0:
            return None, None
        batch = random.sample(self.buffer, min(batch_size, len(self.buffer)))
        feats = torch.stack([x[0] for x in batch])
        lbls  = torch.tensor([x[1] for x in batch])
        return feats, lbls


# ──────────────────────────────────────────────── Dataset
class MyCustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels   = torch.tensor(labels,   dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


# ──────────────────────────────────────────────── Training
def train_on_task(model, train_loader, replay_buffer, task_id,
                  epochs=12, batch_size_replay=64, device="cuda", phase=""):
    
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    prefix = f"[{phase}] " if phase else ""
    
    for epoch in range(epochs):
        total_loss = 0
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            
            replay_feats, replay_lbls = replay_buffer.get_batch(batch_size_replay)
            if replay_feats is not None:
                replay_feats = replay_feats.to(device)
                replay_lbls  = replay_lbls.to(device)
                combined_feats = torch.cat([features, replay_feats], dim=0)
                combined_lbls  = torch.cat([labels,   replay_lbls],  dim=0)
            else:
                combined_feats, combined_lbls = features, labels
            
            optimizer.zero_grad()
            outputs = model(combined_feats)
            loss = criterion(outputs, combined_lbls)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"{prefix}Task {task_id} | Epoch {epoch+1:2d} | Loss: {total_loss/len(train_loader):.4f}")


# ──────────────────────────────────────────────── Evaluation
def evaluate_model(model, loader, device, title="Evaluation"):
    model.eval()
    correct, total = 0, 0
    all_preds, all_true = [], []
    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_true.extend(labels.cpu().numpy())
    
    acc = 100 * correct / total if total > 0 else 0
    print(f"\n{title} Accuracy: {acc:.2f}%  (su {total:,} esempi)")
    print(classification_report(all_true, all_preds,
                                target_names=list(class_map.keys()),
                                digits=4, zero_division=0))


# ──────────────────────────────────────────────── MAIN
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")

    # Seeds per riproducibilità
    seed = 42
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


    # Loading e preprocessing
    df = pd.read_csv('train_test_network.csv')
    df['label_int'] = df['type'].map(class_map)
    df = df.replace('-', 0)
    
    num_cols = [
        'src_port', 'dst_port', 'duration', 'src_bytes', 'dst_bytes', 'missed_bytes',
        'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_qclass', 'dns_qtype',
        'dns_rcode', 'http_request_body_len', 'http_response_body_len', 'http_status_code'
    ]
    num_features = df[num_cols].values.astype(np.float32)
    cat_features = pd.get_dummies(df[['proto', 'conn_state']], dtype=float)
    features = np.hstack((num_features, cat_features.values.astype(np.float32)))
    
    scaler = StandardScaler()
    features = scaler.fit_transform(features)
    labels = df['label_int'].values.astype(np.int64)

    # Split train-test 70/30
    train_features, test_features, train_labels, test_labels = train_test_split(
        features, labels, test_size=0.30, stratify=labels, random_state=42
    )
    print(f"Split → Train: {len(train_labels):,} | Test: {len(test_labels):,} esempi")

    test_dataset = MyCustomDataset(test_features, test_labels)

    
    # ──────────────────────────────────────────────────────────────── FASE 1 – TRAINING INIZIALE
    print("\n" + "="*80)
    print("FASE 1 - TRAINING INIZIALE")
    print("="*80)

    train_dataset_clean = MyCustomDataset(train_features.copy(), train_labels.copy())

    tasks = [[5,8], [9,7], [6,4], [3,2], [1,0]]

    model = SimpleMLP(input_size=features.shape[1], num_classes=10).to(device)
    replay_buffer = ReplayBuffer(max_size=4000)

    for task_id, class_list in enumerate(tasks, 1):
        print(f"\n{'='*60}\n  TASK {task_id}  →  {class_list}\n{'='*60}")

        idx = np.isin(train_dataset_clean.labels.numpy(), class_list)
        train_indices = np.where(idx)[0]
        task_train_dataset = Subset(train_dataset_clean, train_indices)

        train_loader = DataLoader(task_train_dataset, batch_size=128, shuffle=True)

        train_on_task(model, train_loader, replay_buffer, task_id,
                      epochs=12, batch_size_replay=64, device=device, phase="CLEAN")

        replay_size = min(600, len(task_train_dataset))
        if replay_size > 0:
            replay_loader = DataLoader(task_train_dataset, batch_size=replay_size, shuffle=True)
            replay_features, replay_labels = next(iter(replay_loader))
            replay_buffer.add(replay_features, replay_labels, task_id)

        evaluate_model(model, DataLoader(test_dataset, batch_size=256), device,
                       title=f"Dopo Task {task_id}")

    print("\n" + "="*80)
    print("REPORT FINALE DOPO TRAINING INIZIALE")
    print("="*80)
    evaluate_model(model, DataLoader(test_dataset, batch_size=256), device,
                   title="Risultato del training iniziale")


    # ──────────────────────────────────────────────────────────────── FASE 2 – TRAINING AGGIUNTIVO CON POISONED DATA
    print("\n" + "="*80)
    print("FASE 2 - TRAINING AGGIUNTIVO CON POISONED DATA (PERTURBAZIONE + LABEL FLIP MIRATO)")
    print("="*80)

    # ================== PARAMETRI PER I TEST ==================
    porzione_classe_target  = 0.01         # Numero di esempi da usare (1.0 = 100% classe target)
    poison_rate_value       = 1.00         # % di esempi da avvelenare
    classe_da_avvelenare     = 1           # Classe da avvelenare (Es: 1=backdoor)
    classe_destinazione      = 0           # Label dopo il flip (Es: 0=normal)
    rumore_std              = 0.05         # Intensità del rumore gaussiano sulle feature numeriche
    epoche_per_iterazione   = 6            # Epoche per ogni iterazione con dati avvelenati
    num_iterazioni          = 1            # Quante volte ripetere l'avvelenamento
    # ============================================================

    for iterazione in range(1, num_iterazioni + 1):
        print(f"\n--- ITERAZIONE {iterazione}/{num_iterazioni} di training poisoned ---")

        # Selezione di alcuni esempi della classe target
        idx_target = np.where(train_labels == classe_da_avvelenare)[0]
        n_target = len(idx_target)

        if n_target == 0:
            print(f"Errore: nessuna istanza della classe {classe_da_avvelenare} nel training set.")
            continue

        n_porzione = int(n_target * porzione_classe_target)
        idx_porzione = np.random.choice(idx_target, n_porzione, replace=False)

        poisoned_features = train_features[idx_porzione].copy()
        poisoned_labels   = train_labels[idx_porzione].copy()

        print(f"Selezione esempi: {n_porzione:,} / {n_target:,} esempi "
              f"({porzione_classe_target*100:.1f}%) della classe {classe_da_avvelenare}")

        # Aggiunta rumore gaussiano sulle feature numeriche
        n_numeric = len(num_cols)
        poisoned_features[:, :n_numeric] += np.random.normal(
            loc=0.0, scale=rumore_std, size=(n_porzione, n_numeric)
        )

        print(f"Applicato rumore gaussiano σ={rumore_std} su feature numeriche")

        # Label flip
        idx_to_flip = np.where(poisoned_labels == classe_da_avvelenare)[0]
        num_to_flip = int(len(idx_to_flip) * poison_rate_value)
        if num_to_flip > 0:
            flip_idx = np.random.choice(idx_to_flip, num_to_flip, replace=False)
            poisoned_labels[flip_idx] = classe_destinazione
            print(f"→ Flip eseguito su {num_to_flip:,} / {len(idx_to_flip):,} esempi "
                  f"({poison_rate_value*100:.0f}%) della classe {classe_da_avvelenare} → {classe_destinazione}")
        else:
            print("Nessun flip applicato (Porzione troppo piccola o poison_rate=0)")

        train_dataset_poisoned = MyCustomDataset(poisoned_features, poisoned_labels)
        poisoned_loader = DataLoader(train_dataset_poisoned, batch_size=128, shuffle=True)

        # Prosecuzione del training
        train_on_task(model, poisoned_loader, replay_buffer, task_id=f"poisoned_{iterazione}",
                      epochs=epoche_per_iterazione, batch_size_replay=64, device=device, phase="POISONED")

        print("\n" + "="*80)
        print(f"REPORT DOPO ITERAZIONE {iterazione} POISONED")
        print("="*80)
        evaluate_model(model, DataLoader(test_dataset, batch_size=256), device,
                       title=f"Risultato dopo iterazione {iterazione} POISONED")

    print("\n" + "="*80)
    print("REPORT FINALE DOPO TUTTE LE ITERAZIONI POISONED")
    print("="*80)
    evaluate_model(model, DataLoader(test_dataset, batch_size=256), device,
                   title="Risultato finale del poisoning")

main()

Device: cuda
Split → Train: 147,730 | Test: 63,313 esempi

FASE 1 - TRAINING INIZIALE

  TASK 1  →  [5, 8]
[CLEAN] Task 1 | Epoch  1 | Loss: 0.2681
[CLEAN] Task 1 | Epoch  2 | Loss: 0.0477
[CLEAN] Task 1 | Epoch  3 | Loss: 0.0428
[CLEAN] Task 1 | Epoch  4 | Loss: 0.0439
[CLEAN] Task 1 | Epoch  5 | Loss: 0.0421
[CLEAN] Task 1 | Epoch  6 | Loss: 0.0412
[CLEAN] Task 1 | Epoch  7 | Loss: 0.0421
[CLEAN] Task 1 | Epoch  8 | Loss: 0.0407
[CLEAN] Task 1 | Epoch  9 | Loss: 0.0411
[CLEAN] Task 1 | Epoch 10 | Loss: 0.0398
[CLEAN] Task 1 | Epoch 11 | Loss: 0.0394
[CLEAN] Task 1 | Epoch 12 | Loss: 0.0404

Dopo Task 1 Accuracy: 9.78%  (su 63,313 esempi)
              precision    recall  f1-score   support

      normal     0.0000    0.0000    0.0000     15000
    backdoor     0.0000    0.0000    0.0000      6000
        ddos     0.0000    0.0000    0.0000      6000
         dos     0.0000    0.0000    0.0000      6000
   injection     0.0000    0.0000    0.0000      6000
        mitm     0.0105    